# Piloto FinBERT SEC Form 8-K: Pipeline de Extracción y Scoring
## Comparativa Empírica de 6 Variantes: RegEx vs. spaCy (Ventanas 0, ±1, ±2)
**Máster en Data Science — Trabajo de Fin de Máster (TFM)**
*Entorno de Datos: Cloudflare R2 (`raw/sec/8k-2010-2023.parquet`) | Modelo: `ProsusAI/finbert`*

---
### Objetivos del Cuaderno
1. Descargar el dataset curado de hechos relevantes SEC Form 8-K (2010–2023) desde el Data Lake de Cloudflare R2 vía `get_storage_repository()`.
2. Realizar un **muestreo estratificado balanceado de $n = 150$ eventos** representativos entre las 6 principales entidades bancarias (`JPM`, `BAC`, `WFC`, `C`, `GS`, `MS`) y los 5 ítems regulatorios más críticos:
   - `Item 2.02`: Resultados de operaciones y condición financiera ($n=50$).
   - `Item 8.01`: Otros eventos relevantes ($n=40$).
   - `Item 5.02`: Nombramiento y cese de consejeros/directivos ($n=25$).
   - `Item 7.01`: Divulgación Regulation FD ($n=20$).
   - `Item 9.01`: Estados financieros y anexos ($n=15$).
3. Implementar y ejecutar las **6 variantes de extracción guiada**:
   - `RE_V0_PODA`: Regex splitting + Ventana 0 + Poda Boilerplate SEC.
   - `RE_V1_PODA`: Regex splitting + Ventana $\pm 1$ + Poda Boilerplate SEC.
   - `RE_V2_PODA`: Regex splitting + Ventana $\pm 2$ + Poda Boilerplate SEC.
   - `SPACY_S0`: spaCy parsing + Ventana 0 + Poda Boilerplate SEC.
   - `SPACY_S1B`: spaCy parsing + Ventana $\pm 1$ + Poda Boilerplate SEC.
   - `SPACY_S2B`: spaCy parsing + Ventana $\pm 2$ + Poda Boilerplate SEC.
4. Ejecutar la inferencia real con FinBERT sobre las $900$ secuencias resultantes ($150 	imes 6$).
5. Integrar la categorización Ground Truth y consolidar el dataset único en `scratch/finbert_sec_audit_150.csv`.


In [8]:
import asyncio
import io
import json
import re
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import spacy
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from IPython.display import display

# 1. Configuración de rutas hacia 'src'
curr = Path.cwd().resolve()
PROJ_ROOT = next((p for p in [curr, *curr.parents] if (p / "pyproject.toml").exists()), curr)
for p in (PROJ_ROOT / "src", PROJ_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from dependencies.dependencies import get_storage_repository
from shared.tickets import ALIAS_BANCOS
from articulos.models.base_nlp import SENTENCE_SPLIT_REGEX

print(f"[OK] Entorno configurado desde {PROJ_ROOT}")

# 2. Cargar modelos NLP y FinBERT
nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
finbert = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").eval()

lbl = {v.lower(): k for k, v in finbert.config.id2label.items()}
i_pos, i_neg, i_neu = lbl["positive"], lbl["negative"], lbl["neutral"]
print(f"[OK] Modelos cargados exitosamente | FinBERT labels: {finbert.config.id2label}")


[OK] Entorno configurado desde C:\Users\felip\OneDrive\Felipe\Master\TFM\TFM-Master


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 286.32it/s]

[OK] Modelos cargados exitosamente | FinBERT labels: {0: 'positive', 1: 'negative', 2: 'neutral'}


## 1. Descarga desde Cloudflare R2 y Muestreo Estratificado ($n=150$)
Descargamos directamente desde el Data Lake el fichero `raw/sec/8k-2010-2023.parquet` utilizando la abstracción centralizada `get_storage_repository()`.
Garantizamos la estratificación balanceada por tipo de ítem regulatorio con semilla determinista (`random_state=42`).


In [9]:
# Conexión S3 y descarga del dataset de hechos relevantes SEC
async def descargar_datos_sec():
    repo = get_storage_repository()
    try:
        data = await repo.get("raw/sec/8k-2010-2023.parquet")
        return pd.read_parquet(io.BytesIO(data))
    finally:
        await repo.close()

# En Jupyter Notebooks el kernel de IPython ya mantiene un event loop activo.
# Ejecutamos mediante ThreadPoolExecutor para compatibilidad 100% transparente (Jupyter y scripts):
import concurrent.futures

try:
    loop = asyncio.get_running_loop()
    with concurrent.futures.ThreadPoolExecutor() as pool:
        df_raw = pool.submit(asyncio.run, descargar_datos_sec()).result()
except RuntimeError:
    df_raw = asyncio.run(descargar_datos_sec())

print(f"[OK] Dataset SEC 8-K descargado: {len(df_raw):,} registros totales")

# Filtrar registros con contenido sustancial (> 50 caracteres)
df_clean = df_raw[df_raw["content"].fillna("").str.strip().str.len() > 50].copy()

# Muestreo estratificado representativo (n=150)
cuotas = {
    "Item 2.02": 50,
    "Item 8.01": 40,
    "Item 5.02": 25,
    "Item 7.01": 20,
    "Item 9.01": 15,
}

submuestras = []
for item_name, n_sample in cuotas.items():
    sub = df_clean[df_clean["item"] == item_name]
    submuestras.append(sub.sample(n=min(n_sample, len(sub)), random_state=42))

df_150 = pd.concat(submuestras).reset_index(drop=True)
df_150["sample_id"] = range(len(df_150))

print(f"[OK] Muestra estratificada consolidada: {len(df_150)} eventos")
print("Distribución por Banco:")
print(df_150["ticker"].value_counts().to_string())
print("Distribución por Ítem Regulatorio:")
print(df_150["item"].value_counts().to_string())


 INFO     | 2026-11-20 14:18:29.170471 | S3 Storage Repository | Abriendo canal persistente con el almacenamiento de objetos S3 (Bootstrap Phase)...
 INFO     | 2026-11-20 14:18:29.194018 | S3 Storage Repository | Conectado a S3 con éxito. Bucket por defecto: [tfm-master-prediccion-bancos-usa] | Endpoint: https://ab02a9e85571f8564208e420a50be80a.eu.r2.cloudflarestorage.com
 INFO     | 2026-11-20 14:18:30.510011 | S3 Storage Repository | Canales del pool de almacenamiento S3 cerrados de forma limpia.
[OK] Dataset SEC 8-K descargado: 5,392 registros totales
[OK] Muestra estratificada consolidada: 150 eventos
Distribución por Banco:
ticker
JPM    40
WFC    39
BAC    23
C      20
GS     15
MS     13
Distribución por Ítem Regulatorio:
item
Item 2.02    50
Item 8.01    40
Item 5.02    25
Item 7.01    20
Item 9.01    15


## 2. Normalización de Hechos Relevantes SEC: Titular Sintético y Poda de Boilerplate
A diferencia de los artículos periodísticos, los informes 8-K carecen de un titular de prensa explícito.
Diseñamos un **Titular Sintético Ancla**:
$$\text{titular} = \text{"[" + ticker + "] " + item + ": " + item\_description}$$

Asimismo, los informes de la SEC contienen bloques contractuales y firmas legales que contaminan el contexto analítico.
Implementamos la poda estricta de:
1. Bloque de firmas (`SIGNATURES` / `Pursuant to the requirements of the Securities Exchange Act of 1934`).
2. Índices formales (`Table of Contents`).
3. Auto-referencias corporativas (`the Company`, `the Corporation`, `the Registrant`, `the Bank`).


In [10]:
def podar_boilerplate_sec(cuerpo: str) -> str:
    """Elimina bloques de firmas legales y cabeceras de navegación formal."""
    out = (cuerpo or "").strip()
    out = re.split(r"SIGNATURES?|Pursuant to the requirements of the Securities Exchange Act of 1934", out, flags=re.IGNORECASE)[0]
    out = re.sub(r"^(Table of Contents|TABLE OF CONTENTS)", "", out, flags=re.MULTILINE)
    out = re.sub(r"\s+", " ", out).strip()
    return out

def get_sec_aliases(ticker: str) -> list[str]:
    """Retorna los alias del banco enriquecidos con auto-referencias institucionales de la SEC."""
    std = ALIAS_BANCOS.get(ticker, [ticker.lower()])
    sec_terms = ["the company", "the corporation", "the registrant", "the bank"]
    return std + sec_terms

def extraer_regex_sec(titular: str, cuerpo: str, ticker: str, ventana: int = 0) -> tuple[str, bool]:
    """Extractor léxico rápido basado en expresiones regulares y ventana simétrica de oraciones."""
    cuerpo_limpio = podar_boilerplate_sec(cuerpo)
    if not cuerpo_limpio:
        return titular.strip(), False
    oraciones = [s.strip() for s in SENTENCE_SPLIT_REGEX.split(cuerpo_limpio) if s.strip()]
    if not oraciones:
        return titular.strip(), False
    aliases = get_sec_aliases(ticker)
    pat = re.compile(r"\b(" + "|".join(re.escape(a) for a in aliases) + r")\b", re.IGNORECASE)
    hits = [i for i, s in enumerate(oraciones) if pat.search(s)]
    if not hits:
        return f"{titular.strip()} [SEP] {oraciones[0]}", False
    idx = sorted({j for i in hits for j in range(i - ventana, i + ventana + 1) if 0 <= j < len(oraciones)})
    return f"{titular.strip()} [SEP] " + " ".join(oraciones[i] for i in idx), True

def extraer_spacy_sec(titular: str, cuerpo: str, ticker: str, ventana: int = 0) -> tuple[str, bool]:
    """Extractor sintáctico dependiente del parser de spaCy con ventana simétrica."""
    cuerpo_limpio = podar_boilerplate_sec(cuerpo)
    if not cuerpo_limpio:
        return titular.strip(), False
    doc = nlp(cuerpo_limpio)
    oraciones = [s.text.strip() for s in doc.sents if s.text.strip()]
    if not oraciones:
        return titular.strip(), False
    aliases = get_sec_aliases(ticker)
    pat = re.compile(r"\b(" + "|".join(re.escape(a) for a in aliases) + r")\b", re.IGNORECASE)
    hits = [i for i, s in enumerate(oraciones) if pat.search(s)]
    if not hits:
        return f"{titular.strip()} [SEP] {oraciones[0]}", False
    idx = sorted({j for i in hits for j in range(i - ventana, i + ventana + 1) if 0 <= j < len(oraciones)})
    return f"{titular.strip()} [SEP] " + " ".join(oraciones[i] for i in idx), True

VARIANTES = {
    "RE_V0_PODA": lambda t, c, k: extraer_regex_sec(t, c, k, ventana=0),
    "RE_V1_PODA": lambda t, c, k: extraer_regex_sec(t, c, k, ventana=1),
    "RE_V2_PODA": lambda t, c, k: extraer_regex_sec(t, c, k, ventana=2),
    "SPACY_S0":   lambda t, c, k: extraer_spacy_sec(t, c, k, ventana=0),
    "SPACY_S1B":  lambda t, c, k: extraer_spacy_sec(t, c, k, ventana=1),
    "SPACY_S2B":  lambda t, c, k: extraer_spacy_sec(t, c, k, ventana=2),
}
print("[OK] Funciones de extracción configuradas para las 6 variantes de la Liga")


[OK] Funciones de extracción configuradas para las 6 variantes de la Liga


## 3. Extracción de Contextos y Medición de Eficiencia Léxica
Ejecutamos la extracción para los 150 eventos y evaluamos los tiempos de procesamiento (ms/doc) y la distribución de tokens generados.


In [11]:
titulares = [f"[{r.ticker}] {r.item}: {r.item_description}" for r in df_150.itertuples()]
df_150["titular"] = titulares

extr_res = {}
tiempos_extr = {}
for nombre, fn in VARIANTES.items():
    t0 = time.perf_counter()
    extr_res[nombre] = [fn(r.titular, r.content, r.ticker) for r in df_150.itertuples()]
    dt = time.perf_counter() - t0
    tiempos_extr[nombre] = dt * 1000 / len(df_150)
    print(f"[{nombre}] Extracción finalizada en {dt:.3f} s ({tiempos_extr[nombre]:.2f} ms/doc)")


[RE_V0_PODA] Extracción finalizada en 0.054 s (0.36 ms/doc)
[RE_V1_PODA] Extracción finalizada en 0.057 s (0.38 ms/doc)
[RE_V2_PODA] Extracción finalizada en 0.048 s (0.32 ms/doc)
[SPACY_S0] Extracción finalizada en 4.719 s (31.46 ms/doc)
[SPACY_S1B] Extracción finalizada en 3.645 s (24.30 ms/doc)
[SPACY_S2B] Extracción finalizada en 3.659 s (24.39 ms/doc)


## 4. Inferencia en Lote con FinBERT ($900$ Secuencias)
Alimentamos a FinBERT con las secuencias extraídas de cada variante, calculando:
- Probabilidades de clase: $P_{\text{pos}}, P_{\text{neg}}, P_{\text{neu}}$.
- Puntuación continua de sentimiento: $\text{sentiment} = P_{\text{pos}} - P_{\text{neg}} \in [-1, +1]$.
- Convicción analítica: $\text{confidence} = 1 - P_{\text{neu}} \in [0, 1]$.
- Predicción discreta con umbral estándar bursátil $\theta = 0.40$.


In [12]:
@torch.inference_mode()
def puntuar_lote(textos, batch_size=16, max_len=512):
    orden = np.argsort([len(t) for t in textos])
    probs = np.zeros((len(textos), 3), dtype=np.float32)
    for i in range(0, len(textos), batch_size):
        ids = orden[i:i + batch_size]
        batch_txt = [textos[j] for j in ids]
        enc = tokenizer(batch_txt, return_tensors="pt", truncation=True, max_length=max_len, padding=True)
        probs[ids] = torch.softmax(finbert(**enc).logits, dim=-1).numpy()
    sent = probs[:, i_pos] - probs[:, i_neg]
    conf = 1.0 - probs[:, i_neu]
    pred = np.where(sent > 0.40, 1, np.where(sent < -0.40, -1, 0))
    return probs[:, i_pos], probs[:, i_neg], probs[:, i_neu], sent, conf, pred

# Estructurar DataFrame consolidado
df_final = pd.DataFrame({
    "id": df_150["sample_id"],
    "ticker": df_150["ticker"],
    "company_name": df_150["company_name"],
    "item": [r.item for r in df_150.itertuples()],
    "item_description": df_150["item_description"],
    "filing_date": df_150["filing_date"].astype(str).str[:10],
    "titular": df_150["titular"],
})

# Inferencia FinBERT por variante
for nombre, lista in extr_res.items():
    textos = [t for t, _ in lista]
    locs = [l for _, l in lista]
    # Usamos tokenize() + 2 ([CLS], [SEP]) para medir longitud real sin generar warnings de posición > 512
    tok_lens = [len(tokenizer.tokenize(t)) + 2 for t in textos]
    
    t0 = time.perf_counter()
    pos, neg, neu, sent, conf, pred = puntuar_lote(textos, batch_size=16, max_len=512)
    dt = time.perf_counter() - t0
    print(f"[FinBERT {nombre}] Inferencia completada en {dt:.2f} s ({dt*1000/len(textos):.1f} ms/doc)")
    
    df_final[f"{nombre}_texto"] = textos
    df_final[f"{nombre}_tokens"] = tok_lens
    df_final[f"{nombre}_localizado"] = locs
    df_final[f"{nombre}_pos"] = pos.round(4)
    df_final[f"{nombre}_neg"] = neg.round(4)
    df_final[f"{nombre}_neu"] = neu.round(4)
    df_final[f"{nombre}_sentiment"] = sent.round(4)
    df_final[f"{nombre}_confidence"] = conf.round(4)
    df_final[f"{nombre}_pred"] = pred


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1190 > 512). Running this sequence through the model will result in indexing errors


[FinBERT RE_V0_PODA] Inferencia completada en 38.90 s (259.3 ms/doc)
[FinBERT RE_V1_PODA] Inferencia completada en 71.14 s (474.3 ms/doc)
[FinBERT RE_V2_PODA] Inferencia completada en 77.25 s (515.0 ms/doc)
[FinBERT SPACY_S0] Inferencia completada en 69.36 s (462.4 ms/doc)
[FinBERT SPACY_S1B] Inferencia completada en 89.41 s (596.1 ms/doc)
[FinBERT SPACY_S2B] Inferencia completada en 77.23 s (514.9 ms/doc)


## 5. Carga de Ground Truth y Consolidación del Dataset Único
Integramos la auditoría pericial realizada por LLM-as-a-Judge con criterios regulatorios de la SEC y exportamos el fichero plano consolidado a `scratch/finbert_sec_audit_150.csv`.


In [13]:
# Cargar el Ground Truth pericial
gt_path = PROJ_ROOT / "scratch" / "ground_truth_sec_150.json"
if gt_path.exists():
    with open(gt_path, "r", encoding="utf-8") as f:
        gt_data = json.load(f)
    df_gt = pd.DataFrame(gt_data)
    df_final["gt_sentiment"] = df_gt["gt_sentiment"]
    df_final["gt_sentiment_str"] = df_gt["gt_sentiment_str"]
    df_final["gt_reason"] = df_gt["gt_reason"]

out_csv = PROJ_ROOT / "scratch" / "finbert_sec_audit_150.csv"
df_final.to_csv(out_csv, index=False, encoding="utf-8")
print(f"[OK] Dataset consolidado guardado exitosamente en {out_csv} ({len(df_final)} filas, {len(df_final.columns)} columnas)")


[OK] Dataset consolidado guardado exitosamente en C:\Users\felip\OneDrive\Felipe\Master\TFM\TFM-Master\scratch\finbert_sec_audit_150.csv (150 filas, 64 columnas)


## 6. Resumen Preliminar de Eficiencia de Extracción
Visualizamos el resumen comparativo de latencia, tamaño medio en tokens y tasa de contención dentro del límite contextual estándar de 256 tokens.


In [14]:
tabla_resumen = []
for nombre in VARIANTES.keys():
    toks = df_final[f"{nombre}_tokens"]
    conf = df_final[f"{nombre}_confidence"]
    tabla_resumen.append({
        "Variante": nombre,
        "Extracción (ms/doc)": round(tiempos_extr[nombre], 2),
        "Tokens Medios": round(float(toks.mean()), 1),
        "% <= 256 tok": round(100 * float((toks <= 256).mean()), 1),
        "Convicción Media": round(float(conf.mean()), 3),
        "% Localizado": round(100 * float(df_final[f"{nombre}_localizado"].mean()), 1),
    })

pd.DataFrame(tabla_resumen)


,Variante,Extracción (ms/doc),Tokens Medios,% <= 256 tok,Convicción Media,% Localizado
0,RE_V0_PODA,0.36,196.8,76.0,0.101,79.3
1,RE_V1_PODA,0.38,239.1,68.7,0.095,79.3
2,RE_V2_PODA,0.32,251.2,66.7,0.090,79.3
3,SPACY_S0,31.46,175.6,78.0,0.103,79.3
4,SPACY_S1B,24.30,231.2,69.3,0.094,79.3
5,SPACY_S2B,24.39,244.7,67.3,0.090,79.3
